In [1]:
!git clone https://github.com/shhadmann/gnn-bert-music-context.git
%cd gnn-bert-music-context
!pip install torch-geometric transformers librosa scikit-learn pandas seaborn tqdm pyyaml

Cloning into 'gnn-bert-music-context'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 320 (delta 30), reused 48 (delta 12), pack-reused 252 (from 1)
Receiving objects: 100% (320/320), 1.06 MiB | 3.03 MiB/s, done.
Resolving deltas: 100% (140/140), done.
/content/gnn-bert-music-context
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.9 MB/s eta 0:00:00


In [2]:
!mkdir -p /content/gnn-bert-music-context/data/processed/magnatagatune
!mkdir -p /content/gnn-bert-music-context/results
!mv /content/text_labels.csv /content/gnn-bert-music-context/data/processed/magnatagatune/
!mv /content/top50_tags.json /content/gnn-bert-music-context/data/processed/magnatagatune/
!unzip -o -q /content/mtat_graphs_for_colab.zip -d /content/gnn-bert-music-context/data/processed/magnatagatune/
!unzip -o -q /content/cross_attention_checkpoints.zip -d /content/gnn-bert-music-context/results/

In [3]:
!ls /content/gnn-bert-music-context/data/processed/magnatagatune/
!ls /content/gnn-bert-music-context/results/ | grep cross_attention

graphs	text_labels.csv  top50_tags.json
task3_cross_attention_bert.pt
task3_cross_attention_fusion.pt
task3_cross_attention_gnn.pt
task3_cross_attention_test_metrics.json


# GNN-BERT Music Context — End-to-End Inference Demo

This notebook demonstrates the project's core required architecture
(Task 3's GNN-BERT cross-attention fusion model) on a single real
example: loading the trained model, running one MagnaTagATune test
clip through it, and showing the predicted tags.

In [4]:
import sys
sys.path.append('/content/gnn-bert-music-context/src')

import json
import torch
import pandas as pd
import yaml

from bert_encoder import BertTagClassifier, load_tokenizer, tokenize_batch
from gnn_model import GNNGenreClassifier
from fusion_model import CrossAttentionFusion

with open('/content/gnn-bert-music-context/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cpu')
print(f"Using device: {device}")

Using device: cpu


## Load the trained models

Three components: the GNN (encodes the audio segment graph), BERT
(encodes the text), and the cross-attention fusion head (combines
both into a tag prediction).

In [5]:
import os
os.chdir('/content/gnn-bert-music-context')

with open('data/processed/magnatagatune/top50_tags.json') as f:
    top50_tags = json.load(f)
with open('results/task3_cross_attention_test_metrics.json') as f:
    threshold = json.load(f)['threshold_used']

tokenizer = load_tokenizer(config['bert']['model_name'])

graph_dim = config['gnn']['hidden_channels']
gnn = GNNGenreClassifier(in_channels=12, hidden_channels=graph_dim,
                          num_layers=config['gnn']['num_layers'], num_classes=10,
                          dropout=config['gnn']['dropout'])
gnn.load_state_dict(torch.load('results/task3_cross_attention_gnn.pt', map_location=device))
gnn.eval()

bert = BertTagClassifier(config['bert']['model_name'], num_tags=50)
bert.load_state_dict(torch.load('results/task3_cross_attention_bert.pt', map_location=device))
bert.eval()

fusion = CrossAttentionFusion(graph_dim=graph_dim, bert_dim=768, num_tags=50)
fusion.load_state_dict(torch.load('results/task3_cross_attention_fusion.pt', map_location=device))
fusion.eval()

print(f"Models loaded. Tuned classification threshold: {threshold:.2f}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded. Tuned classification threshold: 0.20


## Pick one real test example

Using a MagnaTagATune clip from the held-out test split — this clip
was never seen during training.

In [6]:
from torch_geometric.loader import DataLoader

with open('data/splits/mtag_test.json') as f:
    test_ids = json.load(f)

text_labels = pd.read_csv('data/processed/magnatagatune/text_labels.csv')

# Pick the first test clip that actually has a graph on disk
example_clip_id = None
for cid in test_ids:
    if os.path.exists(f'data/processed/magnatagatune/graphs/{cid}.pt'):
        example_clip_id = cid
        break

graph = torch.load(f'data/processed/magnatagatune/graphs/{example_clip_id}.pt', weights_only=False)
row = text_labels[text_labels['clip_id'] == example_clip_id].iloc[0]
text = row['text']
true_tags = [t for t in top50_tags if row[t] == 1]

print(f"Clip ID: {example_clip_id}")
print(f"Text (title+artist+album): \"{text}\"")
print(f"Ground-truth tags: {true_tags}")

Clip ID: 2
Text (title+artist+album): "BWV54 - I Aria American Bach Soloists J.S. Bach Solo Cantatas"
Ground-truth tags: ['classical', 'strings', 'violin', 'opera']


## Run inference

Graph → GNN encoder → graph embedding.
Text → BERT encoder → token embeddings.
Both → cross-attention fusion → predicted tags.

In [7]:
loader = DataLoader([graph], batch_size=1)
batch = next(iter(loader))

tokenized = tokenize_batch(tokenizer, [text], max_length=config['bert']['max_length'])

with torch.no_grad():
    _, graph_emb = gnn(batch.x, batch.edge_index, batch.batch)
    bert_out = bert.bert(input_ids=tokenized['input_ids'], attention_mask=tokenized['attention_mask'])
    token_emb = bert_out.last_hidden_state
    logits, z, attn_weights = fusion(graph_emb, token_emb, tokenized['attention_mask'])
    probs = torch.sigmoid(logits)[0]

predicted_tags = [top50_tags[i] for i in range(50) if probs[i] > threshold]
correctly_matched = set(predicted_tags) & set(true_tags)

print(f"Predicted tags: {predicted_tags}")
print(f"Correctly matched: {sorted(correctly_matched)}")
print(f"\nFused embedding shape: {z.shape}")

Predicted tags: ['classical', 'strings', 'violin', 'opera']
Correctly matched: ['classical', 'opera', 'strings', 'violin']

Fused embedding shape: torch.Size([1, 256])


## Summary

This demonstrates the full pipeline end-to-end: raw audio → segment
graph (built in Phase 1) → GNN embedding, paired with catalog text →
BERT embedding, combined via cross-attention fusion (Phase 4) →
multi-label tag prediction, using the actual trained checkpoint
(macro-F1=0.317 on the full test set, see `results/master_results_table.md`
for complete metrics).